In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
WORK_DIR="/home/magnolia/DataScience/Stellar_Class"
TRAIN_PATH=Path(WORK_DIR,"data/train.csv")
TEST_PATH=Path(WORK_DIR, "data/test.csv")
MODEL_DIR=Path(WORK_DIR,"models")

In [14]:
df=pd.read_csv(TRAIN_PATH).set_index('id')
target=df['class']
X=df.drop(['class'], axis=1)

In [143]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

class FeatureEngineer:

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self._scalers = {}
        self._encoders = {}



    def minmax_scale(self, columns: list) -> "FeatureEngineer":

        for col in columns:
            scaler = MinMaxScaler()
            self.df[col] = scaler.fit_transform(self.df[[col]])
            self._scalers[col] = scaler
        return self



    def standard_scale(self, columns: list) -> "FeatureEngineer":
        
        for col in columns:
            scaler = StandardScaler()
            self.df[col] = scaler.fit_transform(self.df[[col]])
            self._scalers[col] = scaler
        return self



    def log_transform(self, columns: list) -> "FeatureEngineer":
        
        for col in columns:
            self.df[col] = np.log1p(self.df[col])
        return self



    def onehot_encode(self, columns: list, drop_first: bool = True) -> "FeatureEngineer":
        self.df = pd.get_dummies(self.df, columns=columns, drop_first=drop_first)
        return self



    def equal_width_bin(self, columns: list, bins: int) -> "FeatureEngineer":
        # and create additional features
        for col in columns:
            new_col = f"{col}_bin"
            labels = [f"Q_{i}" for i in range(bins)]
            self.df[new_col] = pd.cut(self.df[col], bins=bins, labels=labels)
        return self


    def ordinal_encode(self, columns: list) -> "FeatureEngineer":

        for col in columns:
            encoder=OrdinalEncoder()
            self.df[col]=encoder.fit_transform(self.df[[col]])
            self._encoders[col] = encoder
        return self



    def get_df(self) -> pd.DataFrame:
        return self.df



    def summary(self) -> None:
        print(f"Shape     : {self.df.shape}")
        print(f"Scaled    : {list(self._scalers.keys()) or 'none'}")
        print(f"Encoded   : {list(self._encoders.keys()) or 'none'}")
        print(f"Dtypes    :\n{self.df.dtypes}")

In [144]:
fe=FeatureEngineer(X)
NUM_COLS=fe.get_df().select_dtypes('float64').columns.to_list()
CAT_COLS=fe.get_df().select_dtypes('object').columns.to_list()

results=(fe.
         minmax_scale(columns=NUM_COLS).
         onehot_encode(columns=CAT_COLS, drop_first=False).
         equal_width_bin(columns=NUM_COLS, bins=30).
         get_df())


In [145]:
results

,alpha,delta,u,g,r,i,z,redshift,spectral_type_A/F,spectral_type_G/K,...,galaxy_population_Blue_Cloud,galaxy_population_Red_Sequence,alpha_bin,delta_bin,u_bin,g_bin,r_bin,i_bin,z_bin,redshift_bin
id,,,,,,,,,,,,,,,,,,,,,
0,0.410354,0.359600,0.902047,0.593556,0.613685,0.457380,0.458150,0.059673,False,False,...,False,True,Q_12,Q_10,Q_27,Q_17,Q_18,Q_13,Q_13,Q_1
1,0.355503,0.518029,0.736735,0.394156,0.395090,0.330026,0.337005,0.023921,False,False,...,False,True,Q_10,Q_15,Q_22,Q_11,Q_11,Q_9,Q_10,Q_0
2,0.499408,0.548897,0.745776,0.535591,0.677899,0.540495,0.586009,0.403624,False,False,...,True,False,Q_14,Q_16,Q_22,Q_16,Q_20,Q_16,Q_17,Q_12
3,0.627261,0.685057,0.825721,0.533575,0.507953,0.401483,0.411524,0.077779,False,False,...,False,True,Q_18,Q_20,Q_24,Q_16,Q_15,Q_12,Q_12,Q_2
4,0.393970,0.384141,0.769301,0.421463,0.446154,0.372250,0.391796,0.080580,False,False,...,False,True,Q_11,Q_11,Q_23,Q_12,Q_13,Q_11,Q_11,Q_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577342,0.620930,0.210766,0.738504,0.377623,0.404234,0.327799,0.321483,0.074279,False,False,...,False,True,Q_18,Q_6,Q_22,Q_11,Q_12,Q_9,Q_9,Q_2
577343,0.621921,0.604748,0.840855,0.626472,0.640505,0.452561,0.479691,0.095226,False,False,...,False,True,Q_18,Q_18,Q_25,Q_18,Q_19,Q_13,Q_14,Q_2
577344,0.145136,0.191905,0.777793,0.545298,0.508601,0.426979,0.430571,0.055024,False,False,...,False,True,Q_4,Q_5,Q_23,Q_16,Q_15,Q_12,Q_12,Q_1
